# Limpieza · demanda real nacional de SIMEM (`14fabb`)

La serie horaria nacional que resulta de sumar el detalle por empresa de SIMEM,
**con una sola versión de liquidación por hora** —sumar todas multiplicaría la
demanda hasta ×5—. Es la contraparte de SIMEM de la demanda real de XM.

Lo que se encontró al diagnosticarla:

- serie completa: 49 776 horas, sin huecos, duplicados ni ceros;
- 146 atípicos estacionales;
- la versión de liquidación cambia con el tiempo: el histórico está en versiones
  ya cerradas, agosto de 2026 en `TXR` y septiembre de 2026 en `TX2`, que es
  preliminar y todavía puede revisarse;
- XM queda en promedio un 1,44 % por encima (alcance distinto de las dos
  series), sin ninguna hora que difiera más de un 10 %.

Se verificó además que en **todas** las horas la versión elegida trae todas las
combinaciones de empresa y mercado, así que el colapso de versiones no deja a
ninguna empresa fuera del total.

La versión preliminar no se trata como un error: la columna `version` se
conserva para que el modelado decida qué hacer con las horas aún revisables.

In [1]:
import sys, warnings
from pathlib import Path

# Sube desde la carpeta del notebook hasta la raiz del proyecto, la que contiene
# src/ingesta: funciona igual en notebooks/ que en notebooks/limpieza/<dataset>/.
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "ingesta").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS = RAIZ / "datasets"
print("raiz     :", RAIZ)
print("datasets :", DATASETS, "->", "existe" if DATASETS.exists() else "FALTA")

raiz     : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER
datasets : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets -> existe


In [2]:
simem = pd.read_csv(DATASETS / "simem" / "simem_14fabb_nacional.csv", parse_dates=["fecha_hora"])
print(f"{len(simem):,} filas · {simem.fecha_hora.min()} .. {simem.fecha_hora.max()}")
simem.head()

49,776 filas · 2021-01-01 00:00:00-05:00 .. 2026-09-05 23:00:00-05:00


,fecha_hora,fuente,metrica,entidad,valor_kwh,version
0,2021-01-01 00:00:00-05:00,simem,DdaReal,Nacional,6515688.69,TX4
1,2021-01-01 01:00:00-05:00,simem,DdaReal,Nacional,6320018.11,TX4
2,2021-01-01 02:00:00-05:00,simem,DdaReal,Nacional,6091515.10,TX4
3,2021-01-01 03:00:00-05:00,simem,DdaReal,Nacional,5909896.26,TX4
4,2021-01-01 04:00:00-05:00,simem,DdaReal,Nacional,5791504.85,TX4


## Diagnóstico

In [3]:
from calidad.diagnostico import diagnosticar, resumen
informe = diagnosticar(simem)
print(resumen(informe))

DIAGNOSTICO DE CALIDAD
Generado    : 2026-09-11T05:28:33
Forma       : agregada (1.0 filas/hora)
Columnas    : tiempo=fecha_hora  valor=valor_kwh
Filas       : 49,776

--------------------------------------------------------------------------
COMPLETITUD TEMPORAL
--------------------------------------------------------------------------
Rango           : 2021-01-01 00:00:00-05:00 .. 2026-09-05 23:00:00-05:00
Horas           : 49,776 de 49,776 esperadas (100.0%)
Huecos          : ninguno
Timestamps rep. : 0
Dias != 24 h    : ninguno

--------------------------------------------------------------------------
VALORES
--------------------------------------------------------------------------
Descriptivos    : n=49,776  media=8,990,226.4  mediana=9,039,908.1
                  min=5,300,396.4  max=12,274,670.4
Nulos           : ninguno en ninguna columna
Ceros/negativos : 0 ceros, 0 negativos
Atipicos IQR    : 0 (0.0%)  fuera de [5,236,339, 12,713,661]
                  criterio global: no d

### Versión de liquidación a lo largo del tiempo

In [4]:
print("horas por año y versión:")
print(pd.crosstab(simem.fecha_hora.dt.year, simem.version).to_string())
print()
print("últimos meses:")
por_mes = simem.groupby(simem.fecha_hora.dt.strftime("%Y-%m")).version.agg(lambda s: dict(s.value_counts()))
print(por_mes.tail(6).to_string())

horas por año y versión:
version     TX2  TX3   TX4   TX5   TX6   TX7  TX8   TXF  TXR
fecha_hora                                                  
2021          0    0  4320  4440     0     0    0     0    0
2022          0  720  4440  2136  1464     0    0     0    0
2023          0    0  2208  2208  4344     0    0     0    0
2024          0    0  1488  2928  1440  2208  720     0    0
2025          0  720  2904  1464     0     0    0  3672    0
2026        120    0     0     0     0     0    0  5088  744

últimos meses:


fecha_hora
2026-04    {'TXF': 720}
2026-05    {'TXF': 744}
2026-06    {'TXF': 720}
2026-07    {'TXF': 744}
2026-08    {'TXR': 744}
2026-09    {'TX2': 120}


## Limpieza

In [5]:
from limpieza.limpiar import limpiar, resumen as resumen_limpieza
limpio, registro = limpiar(simem)
print(resumen_limpieza(registro))

REGISTRO DE LIMPIEZA
Momento : 2026-09-11T05:28:35
Filas   : 49,776 -> 49,776

[normalizar_esquema]  0 filas afectadas
  criterio: nombres a snake_case; marca de tiempo localizada en UTC-5; valor a float64
  columna_tiempo: fecha_hora
  columna_valor: valor_kwh
  zona_horaria: America/Bogota
  zona_ya_presente: True
  columnas_normalizadas_a_texto: ['fuente', 'metrica', 'entidad', 'version']

[deduplicar]  0 filas afectadas
  criterio: una fila por fecha_hora, conservando la ultima segun el orden de llegada
  filas_por_marca_antes: 1.0

[completar_rejilla]  0 filas afectadas
  criterio: rejilla horaria completa; imputacion 'temporal' solo para huecos de hasta 3 h; sin extrapolar en los extremos
  metodo_imputacion: temporal
  max_horas_interpolacion: 3
  columnas_constantes_propagadas: ['entidad', 'fuente', 'metrica']

[marcar_atipicos]  187 filas afectadas
  criterio: atipico = |z modificada CAUSAL frente a (dia de semana, hora)| > 3.5, calculada solo con observaciones anteriores; ati

## Comparación con la demanda real de XM

Las dos series vienen de la misma fuente primaria pero no miden exactamente lo
mismo (XM excluye el alumbrado público, por ejemplo), así que no se espera que
coincidan. Se guarda la diferencia hora a hora en `diferencia_vs_xm_pct`, como
dato, sin umbral ni marca: sirve para ver cuándo se separan.

In [6]:
from ingesta.normalizar import xm_cliente_a_esquema_comun
xm = xm_cliente_a_esquema_comun(pd.read_csv(DATASETS / "xm" / "xm_demareal_sistema.csv", parse_dates=["timestamp"]))

par = limpio.merge(
    xm[["fecha_hora", "valor_kwh"]].rename(columns={"valor_kwh": "xm_kwh"}),
    on="fecha_hora", how="left", validate="one_to_one",
)
assert len(par) == len(limpio), "el merge cambió el número de filas"

limpio["diferencia_vs_xm_pct"] = ((par.xm_kwh - par.valor_kwh) / par.valor_kwh * 100).to_numpy()
diferencia = limpio.diferencia_vs_xm_pct

registro["operaciones"].append({
    "operacion": "anadir_diferencia_vs_xm",
    "criterio": "diferencia_vs_xm_pct = (XM DemaReal - SIMEM) / SIMEM * 100, hora a hora; dato, sin marca",
    "filas_afectadas": 0,
    "detalle": {
        "media": round(float(diferencia.mean()), 4),
        "p1": round(float(diferencia.quantile(.01)), 4),
        "p99": round(float(diferencia.quantile(.99)), 4),
        "horas_sin_xm": int(diferencia.isna().sum()),
    },
})

print(diferencia.describe(percentiles=[.01, .5, .99]).round(3).to_string())
print()
print("diferencia media por año (%):")
print(diferencia.groupby(limpio.fecha_hora.dt.year).mean().round(3).to_string())

count    49776.000
mean         1.439
std          1.689
min          0.107
1%           0.209
50%          0.559
99%          6.402
max          7.692

diferencia media por año (%):
fecha_hora
2021    0.875
2022    1.057
2023    2.149
2024    2.092
2025    1.252
2026    1.093


## Guardar

In [7]:
import json
SALIDA = DATASETS / "limpios"
SALIDA.mkdir(parents=True, exist_ok=True)
limpio.to_parquet(SALIDA / "simem_nacional.parquet", index=False)
with open(SALIDA / "registro_simem_nacional.json", "w", encoding="utf-8") as f:
    json.dump(registro, f, ensure_ascii=False, indent=2, default=str)
print(f"{len(limpio):,} filas -> {SALIDA / 'simem_nacional.parquet'}")

49,776 filas -> C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets\limpios\simem_nacional.parquet
